In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ruta al directorio del proyecto y de los subconjuntos
RUTA_PROYECTO = '/content/drive/MyDrive/Proyecto-neumonia'
RUTA_DATOS = os.path.join(RUTA_PROYECTO, 'Subconjunto-datos')

os.chdir(RUTA_PROYECTO)
print(f"Directorio de trabajo: {os.getcwd()}")
print(f"\nContenido de Subconjunto-datos:")
for item in sorted(os.listdir(RUTA_DATOS)):
    print(f"  {item}")

In [ ]:
# Cargar los .npy originales del Notebook 1
X_train = np.load(os.path.join(RUTA_DATOS, 'X_train.npy'))
y_train = np.load(os.path.join(RUTA_DATOS, 'y_train.npy'))
X_val = np.load(os.path.join(RUTA_DATOS, 'X_val.npy'))
y_val = np.load(os.path.join(RUTA_DATOS, 'y_val.npy'))
X_test = np.load(os.path.join(RUTA_DATOS, 'X_test.npy'))
y_test = np.load(os.path.join(RUTA_DATOS, 'y_test.npy'))

print(f"Resumen de dimensiones (datos originales):")
print(f"  X_train: {X_train.shape} | Rango: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

In [ ]:
# Cargar los .npy originales del Notebook 1
X_train = np.load(os.path.join(RUTA_DATOS, 'X_train.npy'))
y_train = np.load(os.path.join(RUTA_DATOS, 'y_train.npy'))
X_val = np.load(os.path.join(RUTA_DATOS, 'X_val.npy'))
y_val = np.load(os.path.join(RUTA_DATOS, 'y_val.npy'))
X_test = np.load(os.path.join(RUTA_DATOS, 'X_test.npy'))
y_test = np.load(os.path.join(RUTA_DATOS, 'y_test.npy'))

print(f"Resumen de dimensiones (datos originales):")
print(f"  X_train: {X_train.shape} | Rango: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

In [ ]:
SEED = 230406
tf.keras.utils.set_random_seed(SEED)

BATCH_SIZE = 32
IMG_SHAPE = (180, 180, 1)
EPOCHS = 15

def build_cnn_kernel_3x3():
    """CNN con kernel 3x3 (Modelo 1 del Paper 3)"""
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=IMG_SHAPE),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

def build_cnn_kernel_5x5():
    """CNN con kernel 5x5 (Modelo 2 del Paper 3)"""
    model = Sequential([
        Conv2D(32, (5, 5), activation='relu', input_shape=IMG_SHAPE),
        MaxPooling2D(2, 2),
        Conv2D(64, (5, 5), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

def build_cnn_kernel_7x7():
    """CNN con kernel 7x7 (Modelo 3 del Paper 3)"""
    model = Sequential([
        Conv2D(32, (7, 7), activation='relu', input_shape=IMG_SHAPE),
        MaxPooling2D(2, 2),
        Conv2D(64, (7, 7), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

# Mostrar resumen de cada modelo
print("=" * 70)
print("ARQUITECTURAS DEL PAPER 3 (ENSEMBLE DE 3 CNNs)")
print("=" * 70)

model_3x3 = build_cnn_kernel_3x3()
print("\nModelo 1: Kernel 3x3")
model_3x3.summary()

model_5x5 = build_cnn_kernel_5x5()
print("\nModelo 2: Kernel 5x5")
model_5x5.summary()

model_7x7 = build_cnn_kernel_7x7()
print("\nModelo 3: Kernel 7x7")
model_7x7.summary()

In [ ]:
historiales_sin_da = {}
modelos_sin_da = {}

# Modelo 1: Kernel 3x3
print("=" * 70)
print("ENTRENANDO MODELO 1 (Kernel 3x3) - SIN DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_3x3 = build_cnn_kernel_3x3()
historiales_sin_da['Modelo_3x3'] = model_3x3.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)
modelos_sin_da['Modelo_3x3'] = model_3x3

# Modelo 2: Kernel 5x5
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 2 (Kernel 5x5) - SIN DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_5x5 = build_cnn_kernel_5x5()
historiales_sin_da['Modelo_5x5'] = model_5x5.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)
modelos_sin_da['Modelo_5x5'] = model_5x5

# Modelo 3: Kernel 7x7
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 3 (Kernel 7x7) - SIN DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_7x7 = build_cnn_kernel_7x7()
historiales_sin_da['Modelo_7x7'] = model_7x7.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)
modelos_sin_da['Modelo_7x7'] = model_7x7

In [ ]:
historiales_con_da = {}
modelos_con_da = {}

# Modelo 1: Kernel 3x3
print("=" * 70)
print("ENTRENANDO MODELO 1 (Kernel 3x3) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_3x3_da = build_cnn_kernel_3x3()
historiales_con_da['Modelo_3x3_DA'] = model_3x3_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_3x3_DA'] = model_3x3_da

# Modelo 2: Kernel 5x5
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 2 (Kernel 5x5) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_5x5_da = build_cnn_kernel_5x5()
historiales_con_da['Modelo_5x5_DA'] = model_5x5_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_5x5_DA'] = model_5x5_da

# Modelo 3: Kernel 7x7
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 3 (Kernel 7x7) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_7x7_da = build_cnn_kernel_7x7()
historiales_con_da['Modelo_7x7_DA'] = model_7x7_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_7x7_DA'] = model_7x7_da

In [ ]:
historiales_con_da = {}
modelos_con_da = {}

# Modelo 1: Kernel 3x3
print("=" * 70)
print("ENTRENANDO MODELO 1 (Kernel 3x3) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_3x3_da = build_cnn_kernel_3x3()
historiales_con_da['Modelo_3x3_DA'] = model_3x3_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_3x3_DA'] = model_3x3_da

# Modelo 2: Kernel 5x5
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 2 (Kernel 5x5) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_5x5_da = build_cnn_kernel_5x5()
historiales_con_da['Modelo_5x5_DA'] = model_5x5_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_5x5_DA'] = model_5x5_da

# Modelo 3: Kernel 7x7
print("\n" + "=" * 70)
print("ENTRENANDO MODELO 3 (Kernel 7x7) - CON DATA AUGMENTATION")
print("=" * 70)
tf.keras.utils.set_random_seed(SEED)
model_7x7_da = build_cnn_kernel_7x7()
historiales_con_da['Modelo_7x7_DA'] = model_7x7_da.fit(
    X_train_da, y_train_da,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_da, y_val_da),
    verbose=1
)
modelos_con_da['Modelo_7x7_DA'] = model_7x7_da

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# SIN DATA AUGMENTATION
for idx, (nombre, hist) in enumerate(historiales_sin_da.items()):
    epocas = range(1, len(hist.history['loss']) + 1)
    
    # Loss
    axes[0, idx].plot(epocas, hist.history['loss'], label='Train Loss', lw=2, color='tab:blue')
    axes[0, idx].plot(epocas, hist.history['val_loss'], label='Val Loss', lw=2, linestyle='--', color='tab:orange')
    axes[0, idx].set_title(f"{nombre} - Sin DA (Loss)", fontweight='bold')
    axes[0, idx].set_ylabel("Binary Crossentropy")
    axes[0, idx].set_xlabel("Época")
    axes[0, idx].grid(True, alpha=0.3)
    axes[0, idx].legend()
    
    # Accuracy
    axes[1, idx].plot(epocas, hist.history['accuracy'], label='Train Acc', lw=2, color='tab:green')
    axes[1, idx].plot(epocas, hist.history['val_accuracy'], label='Val Acc', lw=2, linestyle='--', color='tab:red')
    axes[1, idx].set_title(f"{nombre} - Sin DA (Accuracy)", fontweight='bold')
    axes[1, idx].set_ylabel("Accuracy")
    axes[1, idx].set_xlabel("Época")
    axes[1, idx].set_ylim(0.5, 1.02)
    axes[1, idx].grid(True, alpha=0.3)
    axes[1, idx].legend()

plt.suptitle("Modelos del Paper 3 - Entrenamiento SIN Data Augmentation",
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# CON DATA AUGMENTATION
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (nombre, hist) in enumerate(historiales_con_da.items()):
    epocas = range(1, len(hist.history['loss']) + 1)
    
    # Loss
    axes[0, idx].plot(epocas, hist.history['loss'], label='Train Loss', lw=2, color='tab:blue')
    axes[0, idx].plot(epocas, hist.history['val_loss'], label='Val Loss', lw=2, linestyle='--', color='tab:orange')
    axes[0, idx].set_title(f"{nombre} - Con DA (Loss)", fontweight='bold')
    axes[0, idx].set_ylabel("Binary Crossentropy")
    axes[0, idx].set_xlabel("Época")
    axes[0, idx].grid(True, alpha=0.3)
    axes[0, idx].legend()
    
    # Accuracy
    axes[1, idx].plot(epocas, hist.history['accuracy'], label='Train Acc', lw=2, color='tab:green')
    axes[1, idx].plot(epocas, hist.history['val_accuracy'], label='Val Acc', lw=2, linestyle='--', color='tab:red')
    axes[1, idx].set_title(f"{nombre} - Con DA (Accuracy)", fontweight='bold')
    axes[1, idx].set_ylabel("Accuracy")
    axes[1, idx].set_xlabel("Época")
    axes[1, idx].set_ylim(0.5, 1.02)
    axes[1, idx].grid(True, alpha=0.3)
    axes[1, idx].legend()

plt.suptitle("Modelos del Paper 3 - Entrenamiento CON Data Augmentation",
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Seleccionar el mejor modelo basado en Recall (Sensibilidad) y F1-Score
mejor_modelo_info = df_resultados.loc[df_resultados['Recall (Sensibilidad)'].idxmax()]

print("=" * 70)
print("MEJOR MODELO SELECCIONADO")
print("=" * 70)
print(f"Modelo: {mejor_modelo_info['Modelo']}")
print(f"Data Augmentation: {mejor_modelo_info['Data Augmentation']}")
print(f" -> Accuracy: {mejor_modelo_info['Accuracy']}")
print(f" -> Precision: {mejor_modelo_info['Precision']}")
print(f" -> Recall (Sensibilidad): {mejor_modelo_info['Recall (Sensibilidad)']}")
print(f" -> F1-Score: {mejor_modelo_info['F1-Score']}")
print(f" -> AUC-ROC: {mejor_modelo_info['AUC-ROC']}")
print(f" -> Falsos Negativos: {mejor_modelo_info['Falsos Negativos (FN)']}")
print(f" -> Brecha Overfitting: {mejor_modelo_info['Brecha Overfitting']}")
print("=" * 70)

# Guardar el mejor modelo
if mejor_modelo_info['Data Augmentation'] == 'No':
    mejor_modelo = modelos_sin_da[mejor_modelo_info['Modelo']]
else:
    mejor_modelo = modelos_con_da[mejor_modelo_info['Modelo']]

ruta_modelo = 'mejor_modelo_notebook05_paper3.keras'
mejor_modelo.save(ruta_modelo)
print(f"\n✅ Modelo guardado en: {os.path.abspath(ruta_modelo)}")

In [ ]:
print("=" * 70)
print("RESUMEN - NOTEBOOK 5: PAPER 3 (ENSEMBLE DE 3 CNNs)")
print("=" * 70)
print(f"\nTotal de modelos entrenados: {len(df_resultados)}")
print(f"  - Sin Data Augmentation: {len(modelos_sin_da)}")
print(f"  - Con Data Augmentation: {len(modelos_con_da)}")
print(f"\nMejor modelo: {mejor_modelo_info['Modelo']}")
print(f"  - Recall: {mejor_modelo_info['Recall (Sensibilidad)']}")
print(f"  - F1-Score: {mejor_modelo_info['F1-Score']}")
print(f"  - AUC-ROC: {mejor_modelo_info['AUC-ROC']}")
print("=" * 70)